# Семинар №5. Decoder: генерация текста, управление распределением токенов и цена модели

В прошлом семинаре мы работали с **BERT**, который строит контекстные представления входного текста. Теперь исследуем модель **decoder-only**: как она вычисляет вероятность следующего токена, выбирает продолжение и почему разные настройки дают разные тексты. В конце сравним ресурсы трёх открытых моделей и вернёмся к задаче NER на архитектуре RoBERTa.

Важное различие: **RoBERTa — encoder-only**, поэтому раздел NER — отдельное закрепление fine-tuning, а не пример работы Decoder.

**Источники для этого занятия:** [The Illustrated GPT-2 — Jay Alammar](https://jalammar.github.io/illustrated-gpt2/), [Harvard — The Annotated Transformer](https://nlp.seas.harvard.edu/annotated-transformer/), [интерактивный Colab: decoder and sampling](https://colab.research.google.com/github/prashantkul/learn-generative-ai/blob/main/07-autoregressive/notebook.ipynb), [Colab: greedy, random, sampling](https://colab.research.google.com/github/book-companion/blog-companion/blob/main/running-models/23-decoding-and-sampling.ipynb). Иллюстрации и код ниже учебные, а не воспроизведение чужих материалов.

## 1. Что именно вычисляет Decoder?

Представьте ввод: **«Сегодня на улице»**. Модель не знает продолжения заранее. Она вычисляет условное распределение вероятностей следующего токена:

$$P(x_{t+1}\mid x_1,\ldots,x_t).$$

Для примера возможны токены «холодно», «тепло», «идёт» и тысячи других. На каждом шаге модель **выдаёт оценки всем токенам словаря** (*logits*), а стратегия декодирования выбирает один токен. Он присоединяется к контексту, затем процесс повторяется. Токен — не обязательно целое слово: это может быть часть слова или знак препинания.

| Шаг | Доступный префикс | Что происходит |
|---|---|---|
| 1 | Сегодня на улице | модель оценивает следующий токен |
| 2 | Сегодня на улице холодно | тот же Decoder получает удлинённый контекст |
| 3 | Сегодня на улице холодно . | модель может продолжить текст или выдать EOS |

Важная формула: $P(x_1,\ldots,x_T)=\prod_{t=1}^T P(x_t\mid x_{<t})$. Это **авторегрессионная факторизация**, а не отдельный способ выбора токенов.

**Как обучают:** при *teacher forcing* модель получает правильные предыдущие токены и учится предсказывать следующий, минимизируя cross-entropy. При **inference** правильное продолжение неизвестно: модель использует собственные предсказания.

### 1.1. Что находится внутри decoder-only Transformer?

Входные идентификаторы токенов превращаются в векторы (*token embeddings*). Позиционная информация позволяет модели учитывать порядок. Далее несколько Transformer-блоков последовательно преобразуют представления:

**Токены → embeddings + информация о позиции → [Causal Self-Attention → Feed Forward] × N → LM head → logits → выбор токена.**

- **Causal Self-Attention**: позиция может учитывать себя и предшествующие токены, но не будущие. Механизм вычисляет Q/K/V из входных представлений.
- **Feed Forward (MLP)**: нелинейно преобразует каждый вектор токена; напрямую не смешивает информацию между позициями.
- **Residual connections и LayerNorm/RMSNorm**: помогают передаче информации и устойчивой оптимизации. Порядок нормализации зависит от модели.
- **LM head**: преобразует последнее скрытое представление в оценки по всему словарю. Это **не** готовые вероятности.

Decoder-only модель отличается от Decoder внутри классического Encoder–Decoder Transformer: последнему также доступен **cross-attention к выходу Encoder**. GPT-подобная модель отдельного Encoder не имеет.

Подробнее: [Illustrated GPT-2](https://jalammar.github.io/illustrated-gpt2/) и [Annotated Transformer](https://nlp.seas.harvard.edu/annotated-transformer/).

### 1.2. Query, Key, Value (Q/K/V): как токены используют контекст

**Откуда берутся Q, K и V?** В предыдущих семинарах мы узнали, что каждый токен представлен вектором. Но одного такого вектора мало, чтобы вычислить, **насколько сильно текущая позиция должна учитывать остальные**.

В Self-Attention из матрицы входных векторов $X$ создаются три проекции с разными обучаемыми весами:

$$Q=XW_Q,\qquad K=XW_K,\qquad V=XW_V.$$

| Вектор | Интуиция | Роль в вычислении |
|---|---|---|
| **Query (Q)** | «Что ищет эта позиция?» | Сравнивается с ключами других позиций |
| **Key (K)** | «По каким признакам меня можно найти?» | Помогает вычислить оценку совместимости с Q |
| **Value (V)** | «Какую информацию передать?» | Участвует во взвешенной сумме |

Эти описания — **аналогия**, а не заранее заданные семантические значения координат. Матрицы $W_Q,W_K,W_V$ обучаются вместе с моделью; их никто не размечает вручную. В **Self-Attention** все три проекции происходят из одной последовательности, а в **Cross-Attention** запросы и ключи/значения могут происходить из разных последовательностей.

Для Decoder добавляется ограничение: позиция $t$ может учитывать только позиции $\le t$.

### 1.3. Пошагово считаем Attention — без большой модели

Предположим, у нас три токена: **«Сегодня / идёт / дождь»**. Для каждого есть один Q, один K, один V. Реальный Transformer вычисляет их обучаемыми линейными слоями, а для учебного опыта используем маленькие заранее заданные векторы.

**Этап 1.** Вычислить оценки $S=QK^\top/\sqrt{d_k}$. Большая оценка — более сильное соответствие между запросом строки и ключом столбца.

**Этап 2.** Запретить будущие позиции: поставить $-\infty$ там, куда Decoder не может смотреть.

**Этап 3.** Применить `softmax` **к каждой строке** и получить веса внимания $A$; их сумма в каждой строке равна 1.

**Этап 4.** Получить контекстные представления: $Z=AV$. Даже если отдельные элементы $V$ не изменились, новый вектор $Z_t$ может использовать информацию из **нескольких предыдущих токенов**.

$$\mathrm{Attention}(Q,K,V)=\mathrm{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}+M\right)V.$$

**Проверьте перед запуском:** какова форма матрицы оценок для 3 токенов? Почему в первой строке causal attention только один ненулевой вес?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

words = ['Сегодня', 'идёт', 'дождь']
Q = np.array([[1., 0.], [0., 1.], [1., 1.]])
K = np.array([[1., 0.], [0., 1.], [1., 1.]])
V = np.array([[1., 2.], [3., 1.], [0., 4.]])
d_k = K.shape[-1]
scores = (Q @ K.T) / np.sqrt(d_k)
allowed = np.tril(np.ones_like(scores, dtype=bool))
masked_scores = np.where(allowed, scores, -np.inf)
exp_scores = np.exp(masked_scores - np.max(masked_scores, axis=-1, keepdims=True))
weights = exp_scores / exp_scores.sum(axis=-1, keepdims=True)
context = weights @ V
print('Q, K, V:', Q.shape, K.shape, V.shape)
print('Матрица оценок:', scores.shape, '| контекст:', context.shape)
print('Суммы весов по строкам:', np.round(weights.sum(axis=1), 5))
fig, ax = plt.subplots(figsize=(6, 3.7))
im = ax.imshow(weights, vmin=0, vmax=1, cmap='Blues')
ax.set_xticks(range(len(words)), words)
ax.set_yticks(range(len(words)), words)
ax.set_xlabel('Key: какую позицию учитываем')
ax.set_ylabel('Query: текущая позиция')
for i in range(len(words)):
    for j in range(len(words)):
        ax.text(j, i, f'{weights[i, j]:.2f}', ha='center', va='center')
fig.colorbar(im, ax=ax, label='Вес внимания')
plt.show()
# TODO: измените только V для слова «дождь» и сравните weights и context.
# TODO: объясните, почему веса не изменятся, а результирующие векторы могут измениться.

**Что важно увидеть:** Q и K определяют **веса**, а V — данные для смешивания. Если изменить только V при неизменных Q и K, матрица весов останется прежней, но результат $Z$ может измениться. В реальной модели проекции связаны общим входом, поэтому изменение самого текста обычно меняет все три.

### 1.4. Почему Multi-Head Attention — это не один Q/K/V

Одна голова внимания имеет собственные $W_Q,W_K,W_V$. В **Multi-Head Attention** такие наборы существуют параллельно; полученные контекстные представления объединяются и проходят выходную проекцию. Разные головы *могут* обучаться разным закономерностям, но это не гарантирует, что каждой голове соответствует понятная лингвистическая функция.

Для кеширования ниже особенно важно: **K и V сохраняются отдельно для каждой головы и для каждого слоя**, а не в виде одного общего вектора для всего предложения.

Дополнительно: [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/) · [The Annotated Transformer](https://nlp.seas.harvard.edu/annotated-transformer/).

### 1.2. Causal mask: как Decoder не «подглядывает» при обучении

При параллельной обработке тренировочной последовательности все токены технически находятся в памяти. Но для предсказания слова на позиции 3 не должны использоваться позиции 4, 5 и далее. **Causal mask** исключает будущие позиции из внимания (добавляя $-\infty$ перед softmax).

В матрице ниже **строка — токен, который обращает внимание; столбец — токен, на который смотрят**. Для `t4` доступны `t1…t4`, а для `t1` — только `t1`.

**Сначала предположите:** сколько единиц будет в строке 4 для 6 токенов? Сколько всего разрешённых пар у последовательности длины 6?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

n_tokens = 6
mask = np.tril(np.ones((n_tokens, n_tokens), dtype=int))
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(mask, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(n_tokens), [f't{i+1}' for i in range(n_tokens)])
ax.set_yticks(range(n_tokens), [f't{i+1}' for i in range(n_tokens)])
ax.set_xlabel('Key: на кого смотрим')
ax.set_ylabel('Query: кто смотрит')
for (i, j), value in np.ndenumerate(mask):
    ax.text(j, i, str(value), ha='center', va='center', color='black')
plt.show()
print('Разрешённых связей:', mask.sum())
# TODO: поменяйте n_tokens на 3 и 8. Выведите, как меняется сумма единиц.

**Интерпретация:** ответ для четвёртой строки — **4**, а сумма разрешённых позиций для длины 6 равна $6\cdot7/2=21$. При росте длины число пар внимания растёт квадратично. Однако в реальной генерации **KV-cache** сохраняет прошлые K/V: на новом шаге не требуется заново вычислять K/V для прежних токенов. Кеш ускоряет decoding, но занимает память, которая растёт с длиной контекста. [Hugging Face: caching](https://huggingface.co/docs/transformers/main/cache_explanation).

## 2. Подготовка среды

Если доступен GPU, выберите **Среда выполнения → Сменить среду выполнения → T4 GPU**. Для исследования генерации используется компактный русскоязычный **ruGPT3-small**; другие модели далее изучаются **только по конфигурациям**, без загрузки весов. Установка приведена одной ячейкой, чтобы версии библиотек совпадали.

In [ ]:
%pip -q install "transformers>=4.46,<5" "datasets>=3,<5" "accelerate>=1,<2" "seqeval>=1.2" "evaluate>=0.4" scikit-learn pandas matplotlib

In [ ]:
import random, time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from transformers import (AutoTokenizer, AutoModelForCausalLM, AutoConfig,
                          AutoModelForTokenClassification, DataCollatorForTokenClassification,
                          TrainingArguments, Trainer, pipeline)
from datasets import load_dataset
from accelerate import init_empty_weights
from sklearn.metrics import classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Устройство:', DEVICE)
if torch.cuda.is_available():
    print('VRAM (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))

# Практика 1. Генерация: какой токен выберет модель?

### Шаг 1. Загрузите одну небольшую модель

Модель возвращает logits — **ненормированные оценки** каждого токена словаря. Для дальнейшего сравнения используем один префикс, одну модель и одинаковое ограничение на число токенов. Предобученная модель не обязана давать качественный диалоговый ответ: здесь мы изучаем механизм, а не оцениваем интеллект модели.

In [ ]:
GEN_MODEL = 'ai-forever/rugpt3small_based_on_gpt2'
try:
    gen_tok = AutoTokenizer.from_pretrained(GEN_MODEL)
    gen_model = AutoModelForCausalLM.from_pretrained(GEN_MODEL).to(DEVICE).eval()
except Exception as exc:
    print('Ошибка загрузки: проверьте интернет / версию библиотек:', exc)
    raise
print('Размер словаря:', len(gen_tok))
print('Параметров модели:', f'{sum(p.numel() for p in gen_model.parameters()):,}')

### Шаг 2. Из logits получить вероятности

Последний вектор `outputs.logits[0, -1, :]` содержит оценки следующего токена для всего словаря. **Softmax** превращает их в вероятности, сумма которых равна 1. Сравним пять кандидатов с наибольшими вероятностями: в зависимости от токенизатора элементы списка могут быть частями слов.

**Прогноз:** сумма показанных top-5 вероятностей обязана быть равна 1? Почему?

In [ ]:
PROMPT = 'Вчера вечером я'
encoded = gen_tok(PROMPT, return_tensors='pt').to(DEVICE)
with torch.no_grad():
    output = gen_model(**encoded)
logits = output.logits[0, -1]
probs = torch.softmax(logits, dim=-1)
values, ids = torch.topk(probs, k=5)
labels = [repr(gen_tok.decode([i])) for i in ids.tolist()]
fig, ax = plt.subplots(figsize=(8, 3))
ax.barh(labels[::-1], values.cpu().numpy()[::-1])
ax.set_xlabel('Вероятность следующего токена')
ax.set_title('Топ-5 кандидатов после prompt')
plt.tight_layout(); plt.show()
print('Сумма вероятностей всего словаря:', probs.sum().item())
print('Сумма top-5:', values.sum().item())
# TODO: попробуйте PROMPT = 'Однажды в лесу' и сравните лидирующие токены.

### Шаг 3. Greedy decoding и sampling

**Greedy search** на каждом шаге берёт `argmax` и при одинаковом вводе обычно воспроизводит тот же результат. Он **локально оптимален**, но не гарантирует наиболее вероятную полную последовательность.

**Sampling** выбирает токен случайно по распределению: результат зависит от `seed`, `temperature` и фильтров. **Temperature** $T$ масштабирует logits перед softmax: $P_i(T)=\mathrm{softmax}(z_i/T)$. При $T<1$ распределение острее, при $T>1$ — более плоское. `T=0` не используют в формуле; для greedy применяют `do_sample=False`.

**Top-k** ограничивает выбор k кандидатами с наибольшими оценками. **Top-p (nucleus sampling)** сохраняет минимальное множество наиболее вероятных токенов, чья накопленная вероятность достигает порога p, и заново нормирует их вероятности. [Hugging Face: decoding strategies](https://huggingface.co/docs/transformers/v4.49.0/generation_strategies).

In [ ]:
settings = {
    'greedy': {'do_sample': False},
    'temperature_0.7': {'do_sample': True, 'temperature': 0.7},
    'top_k_30': {'do_sample': True, 'top_k': 30},
    # TODO: добавьте 'top_p_0.9' с do_sample=True и top_p=0.9
}
rows = []
for name, params in settings.items():
    torch.manual_seed(SEED)
    t0 = time.perf_counter()
    with torch.no_grad():
        result = gen_model.generate(**encoded, max_new_tokens=25,
                                    pad_token_id=gen_tok.eos_token_id,
                                    **params)
    new_ids = result[0, encoded['input_ids'].shape[-1]:]
    rows.append({'strategy': name, 'new_tokens': len(new_ids),
                 'seconds': round(time.perf_counter()-t0, 2),
                 'continuation': gen_tok.decode(new_ids, skip_special_tokens=True)})
pd.DataFrame(rows)

**Проверьте на числах:** сколько новых токенов было создано, одинаково ли время выполнения, что изменилось в тексте? Повторите sampling с другим `seed` — в отличие от greedy, результат может измениться. Не делайте вывод о «качестве» по одному короткому примеру.

### Шаг 4. Beam search — поиск нескольких продолжений

При `num_beams=3` алгоритм на каждом шаге хранит несколько частичных гипотез и сравнивает их **накопленные log-probabilities**. В отличие от sampling, beam search — детерминированный поиск при фиксированных настройках. Он может предпочитать более типовые, иногда повторяющиеся формулировки. Обычно для открытого творческого продолжения sampling полезнее, а для ограниченной генерации beam search может быть уместен.

**Прогноз:** должно ли `num_beams=3` гарантированно дать текст лучше, чем greedy?

In [ ]:
with torch.no_grad():
    beams = gen_model.generate(**encoded, max_new_tokens=25,
                               num_beams=3, do_sample=False,
                               pad_token_id=gen_tok.eos_token_id)
print(gen_tok.decode(beams[0], skip_special_tokens=True))
# TODO: сравните с greedy и объясните, почему качество нельзя оценить только по длине.

# Практика 2. Logits processors: что происходит ДО выбора токена?

Термин **logits processing** означает преобразование оценок или ограничение множества допустимых токенов во время декодирования. `LogitsProcessor` изменяет scores, а `LogitsWarper` (в версиях библиотеки, где он выделен отдельно) реализует такие преобразования, как temperature, top-k и top-p. Это **не постобработка готового текста**: изменения происходят до выбора токена. Постобработка самого текста — отдельный этап (удаление специальных токенов, форматирование, проверка ограничений).

Введём **repetition penalty**: он меняет оценки уже встречавшихся токенов, поэтому иногда уменьшает повторы, но может испортить текст. `no_repeat_ngram_size` запрещает повторять n-граммы заданной длины. [Документация LogitsProcessor](https://huggingface.co/docs/transformers/main/internal/generation_utils).

### Шаг 1. Посмотрим на распределение до и после запрета кандидата

Если сделать logit одного кандидата равным $-\infty$, его вероятность после softmax станет 0. Оставшиеся вероятности перераспределятся между разрешёнными кандидатами. Это показывает, почему ограничение словаря влияет на вероятности **всех** остальных кандидатов.

In [ ]:
toy_logits = torch.tensor([3., 2., 1., 0.])
labels = ['A', 'B', 'C', 'D']
before = torch.softmax(toy_logits, dim=-1)
modified = toy_logits.clone()
modified[0] = -torch.inf
after = torch.softmax(modified, dim=-1)
x = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(x-0.17, before.numpy(), width=0.34, label='Исходно')
ax.bar(x+0.17, after.numpy(), width=0.34, label='A запрещён')
ax.set_xticks(x, labels); ax.set_ylim(0, 1)
ax.set_ylabel('Probability'); ax.legend()
plt.show()
print(pd.DataFrame({'token': labels, 'before': before.numpy(), 'after': after.numpy()}))
# TODO: запретите токен B вместо A. На сколько изменилась вероятность C?

### Шаг 2. Как температура меняет распределение

Это **контролируемый эксперимент с искусственными logits**, поэтому результат легко интерпретировать математически, не путая его со случайностью генерации. При $T=1$ распределение без изменений; при $T=0.5$ лидер усиливается, а при $T=1.5$ вероятности выравниваются.

**Вычислите:** долю самого вероятного кандидата и энтропию $H=-\sum_i p_i\log p_i$ при разных температурах.

In [ ]:
temperatures = [0.5, 1.0, 1.5]
records = []
fig, ax = plt.subplots(figsize=(7, 3))
for T in temperatures:
    p = torch.softmax(toy_logits / T, dim=-1)
    entropy = -(p * p.log()).sum().item()
    records.append({'T': T, 'p_max': p.max().item(), 'entropy': entropy})
    ax.plot(labels, p.numpy(), marker='o', label=f'T={T}')
ax.set_ylabel('P(token)'); ax.legend(); plt.show()
pd.DataFrame(records)
# TODO: добавьте T=2.0 и объясните изменение энтропии.

### Шаг 3. Настоящая модель: штраф за повторения

Сравните **один и тот же prompt** при `repetition_penalty=1.0` и `1.2`. Используйте детерминированный greedy, чтобы контролировать случайность. Сравните повторяющиеся слова или фрагменты; если повторов нет, эффект может быть незаметен — это тоже результат.

In [ ]:
repetition_rows = []
for penalty in [1.0, 1.2]:
    with torch.no_grad():
        out = gen_model.generate(**encoded, max_new_tokens=40,
                                 do_sample=False, repetition_penalty=penalty,
                                 pad_token_id=gen_tok.eos_token_id)
    repetition_rows.append({'penalty': penalty,
                            'text': gen_tok.decode(out[0], skip_special_tokens=True)})
pd.DataFrame(repetition_rows)
# TODO: попробуйте no_repeat_ngram_size=3. Сравните результаты.

### Итог по генерации

Запишите **фактические** результаты:

| Опыт | Какой параметр изменили? | Что изменилось численно? | Что изменилось в тексте? |
|---|---|---|---|
| Greedy vs sampling | do_sample | число токенов / время | ... |
| Temperature | T | p_max и entropy | ... |
| Запрет токена | -inf | P(A), P(B) | в учебном опыте текст не генерируется |
| Repetition penalty | 1.0 vs 1.2 | повторяющиеся фрагменты | ... |

Не путайте **вероятность токена** и **вероятность целого текста**, **logits processing** и **постобработку строки**.

# Практика 3. KV-cache: почему генерация может быть быстрее, но требует памяти

## 3.1. Зачем нужен кеш, если мы уже умеем вычислять Attention?

Возьмём префикс **«Сегодня на улице»**. После добавления нового токена **«холодно»** модель должна предсказать следующий. При **полном повторном прогоне** все предыдущие токены снова участвуют в вычислении Q/K/V на каждом слое. Однако при *causal attention* представления старых позиций **не зависят от будущих токенов**. Поэтому уже вычисленные **K и V** можно сохранить и использовать повторно.

- **Prefill:** обработать все токены исходного запроса за один проход и заполнить кеш K/V.
- **Decode:** передавать только очередной новый токен; рассчитать для него Q/K/V и добавить новые K/V в кеш.
- **Attention при decode:** Query нового токена сравнивается с **прошлыми и новым Key**; по полученным весам объединяются соответствующие Value.

**Почему в названии KV нет Q?** Старые Q не нужны для вычисления представления следующей позиции: у нового токена есть собственный новый Query; зато ему нужны **все доступные предыдущие K/V**. KV-cache **не сохраняет готовый ответ**, а хранит промежуточные тензоры внимания.

| Без кеша | С кешем |
|---|---|
| Повторно обрабатываются все токены префикса | После prefill обрабатывается новый токен |
| Меньше памяти под историю K/V | K/V хранятся в памяти и растут с длиной контекста |
| Больше повторных вычислений | Обычно быстрее последовательная генерация |

KV-cache обычно нужен **для inference**, а не для стандартного полного обучения с teacher forcing, когда все позиции вычисляются параллельно.

## 3.2. Посмотрите на формы K/V внутри готовой модели

Для большинства обычных decoder-only Transformer форма кеша в одном слое приблизительно такова:

$$[\text{batch},\ \text{KV-heads},\ \text{sequence length},\ \text{head dimension}].$$

Два тензора — **Keys** и **Values** — сохраняются **на каждом слое**. В моделях с **Grouped-Query Attention (GQA)** число KV-heads может быть меньше числа Query-heads. В современных реализациях также встречаются sliding-window и другие стратегии, поэтому эту формулу нужно воспринимать как базовую модель для полного кеша.

Начнём с уже загруженной `gen_model`, выполним один `forward` с `use_cache=True` и посмотрим, что вернула библиотека. В зависимости от версии `transformers` кеш может быть представлен объектом `Cache` или совместимым контейнером.

In [ ]:
cache_input = gen_tok('Сегодня на улице', return_tensors='pt').to(DEVICE)
with torch.inference_mode():
    cache_output = gen_model(**cache_input, use_cache=True)
cache = cache_output.past_key_values
print('Тип кеша:', type(cache).__name__)
print('Количество слоёв кеша:', len(cache))
# Для ruGPT3-small и большинства моделей: cache[0] содержит (key, value) первого слоя.
first_key, first_value = cache[0][:2]
print('K первого слоя:', tuple(first_key.shape))
print('V первого слоя:', tuple(first_value.shape))
print('Длина исходного запроса (токенов):', cache_input['input_ids'].shape[1])
# TODO: объясните, какой индекс формы K/V отвечает за длину контекста.
# TODO: проверьте, совпадают ли формы K и V.

**Проверка понимания.** Почему в кеш попадают тензоры **каждого слоя**, а не только последнего? Какая размерность K/V будет увеличиваться по мере генерации? Для ruGPT3-small изучаем конкретный результат выполнения ячейки, не угадываем размеры заранее.

## 3.3. Практический опыт: `use_cache=True` и `use_cache=False`

Hugging Face умеет управлять кешем внутри `model.generate()`. Для сравнения используйте **один prompt, одну модель, greedy-декодирование и одинаковую длину продолжения**. Первый запуск может включать прогрев; лучше сравнить несколько повторов. На CPU разница может быть небольшой или нестабильной — это тоже результат эксперимента, а не ошибка.

Важный нюанс: время генерации зависит не только от алгоритма, но и от GPU, загрузки Colab, длины последовательности и реализации kernels. **Не ожидайте заранее заданного ускорения**.

In [ ]:
import time
prompt_for_cache = 'В ближайшие годы технологии искусственного интеллекта'
cache_inputs = gen_tok(prompt_for_cache, return_tensors='pt').to(DEVICE)

def measure_generation(use_cache, max_new_tokens=40):
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.inference_mode():
        output_ids = gen_model.generate(
            **cache_inputs,
            do_sample=False,
            use_cache=use_cache,
            max_new_tokens=max_new_tokens,
            pad_token_id=gen_tok.eos_token_id,
        )
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    return time.perf_counter() - start, gen_tok.decode(output_ids[0], skip_special_tokens=True)

rows = []
for flag in [False, True]:
    seconds, generated = measure_generation(flag)
    rows.append({'use_cache': flag, 'seconds': seconds})
    print('Кеш включён:', flag, '| секунды:', round(seconds, 3))
    print('Продолжение:', generated[-180:])
print('Замеры:', rows)
# TODO: повторите каждый запуск 3 раза и сравните медианное время, а не один запуск.
# TODO: увеличьте max_new_tokens до 80 и объясните, почему эффект может измениться.

## 3.4. Цена ускорения: сколько памяти занимает KV-cache?

Для **обычного полного KV-cache без сжатия и оконного внимания** теоретическая оценка:

$$\mathrm{KV\ bytes}=2\cdot L\cdot B\cdot H_{kv}\cdot T\cdot D_{head}\cdot b,$$

где $L$ — число слоёв, $B$ — batch size, $H_{kv}$ — количество KV-голов, $T$ — длина контекста, $D_{head}$ — размерность головы, $b$ — байт на одно значение (например, 2 для FP16/BF16). Множитель **2** — отдельно K и V.

Это оценка *только кеша*, а не общего расхода VRAM: веса модели, активации, временные буферы и служебные структуры считаются отдельно. Для GQA используйте именно $H_{kv}$, а не количество Query-голов.

**Прогноз:** если удвоить длину контекста $T$, во сколько раз вырастет базовый KV-cache? А если удвоить batch size?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Учебная конфигурация, не паспортные характеристики ruGPT3-small.
layers = 12
kv_heads = 12
head_dim = 64
batch = 1
bytes_per_value = 2
context_lengths = [128, 256, 512, 1024, 2048]

def kv_cache_mib(seq_len):
    bytes_total = 2 * layers * batch * kv_heads * seq_len * head_dim * bytes_per_value
    return bytes_total / 2**20

memory_mib = [kv_cache_mib(t) for t in context_lengths]
fig, ax = plt.subplots(figsize=(7, 3.7))
ax.bar([str(t) for t in context_lengths], memory_mib)
ax.set(xlabel='Длина контекста, токены', ylabel='Оценка памяти KV-cache, MiB',
       title='Линейный рост KV-cache')
plt.show()
for t, mib in zip(context_lengths, memory_mib):
    print(f'{t:>4} токенов: {mib:.1f} MiB')
# TODO: поменяйте batch на 4 и сравните полученные значения.
# TODO: поменяйте kv_heads на 4 (аналогия с GQA) и объясните экономию памяти.

### Что нужно показать по Q/K/V и KV-cache

1. **Матрица внимания:** показать значения весов, объяснить нули над диагональю и продемонстрировать, что суммы строк равны единице.
2. **Реальный кеш:** назвать фактическую форму K и V первого слоя и объяснить значение каждого измерения.
3. **Скорость:** показать два времени для `use_cache=True/False`; объяснить, почему единичный замер не даёт надёжного вывода.
4. **Память:** назвать размер кеша для 1024 токенов и объяснить, как он изменится при увеличении batch в четыре раза.

Дополнительно: [Hugging Face — How caching works](https://huggingface.co/docs/transformers/main/cache_explanation), [Cache strategies](https://huggingface.co/docs/transformers/main/kv_cache).

# Практика 4. Сколько параметров у Decoder и сколько памяти нужно для fine-tuning?

В реальных проектах перед обучением модели важно оценить требования к GPU. Сравним **три открытых decoder-only модели** от небольшого размера до 1.5B:

1. [HuggingFaceTB/SmolLM2-135M](https://huggingface.co/HuggingFaceTB/SmolLM2-135M)
2. [HuggingFaceTB/SmolLM2-360M](https://huggingface.co/HuggingFaceTB/SmolLM2-360M)
3. [Qwen/Qwen2.5-1.5B](https://huggingface.co/Qwen/Qwen2.5-1.5B)

**Здесь не нужно скачивать веса трёх моделей и тем более запускать их обучение.** Сравниваем архитектурные конфигурации. Открытая модель не обязательно означает разрешение на любые виды коммерческого использования: при реальном проекте проверьте license в model card.

### Три способа оценки

1. **Model card:** узнать опубликованный размер модели и условия использования; это ориентир, округлённые «135M/360M/1.5B» — не точный счёт.
2. **Конфигурация + meta device:** `AutoConfig.from_pretrained()` и `init_empty_weights()` позволяют создать **структуру** модели без размещения весов в RAM/VRAM, затем посчитать `numel()` параметров. Это счёт по текущей конфигурации библиотеки; учитывайте разделяемые веса.
3. **Практический контроль:** после загрузки весов небольшого checkpoint можно посчитать `model.num_parameters()` и `sum(p.numel() ...)`, а потребление GPU — через `torch.cuda.max_memory_allocated()`. Это требует ресурсов; в данном занятии выполняется только для **уже загруженной ruGPT3-small**.

Источники: [Hugging Face — Big Model Initialization](https://huggingface.co/docs/accelerate/main/en/package_reference/big_modeling), [Model training anatomy](https://huggingface.co/docs/transformers/v4.37.0/en/model_memory_anatomy).

In [ ]:
MODEL_IDS = [
    'HuggingFaceTB/SmolLM2-135M',
    'HuggingFaceTB/SmolLM2-360M',
    'Qwen/Qwen2.5-1.5B',
]
for model_id in MODEL_IDS:
    cfg = AutoConfig.from_pretrained(model_id)
    print(model_id, '| type:', cfg.model_type,
          '| layers:', cfg.num_hidden_layers,
          '| hidden:', cfg.hidden_size,
          '| vocab:', cfg.vocab_size)
# TODO: найдите в model cards точное или округлённое число параметров.

### Шаг 2. Подсчёт числа параметров без скачивания весов

`init_empty_weights()` создаёт тензоры на `meta`-устройстве. Их формы известны, но сами численные значения не выделяются. `numel()` возвращает произведение размерностей тензора — количество элементов. Это более надёжно для архитектуры, чем оценивать размер по имени модели.

Для точного учёта tied weights используем уникальные объекты параметров; `model.parameters()` уже обычно возвращает каждый общий параметр один раз. Если модель в meta режиме не поддерживается текущей версией Transformers, пользуйтесь model card и зафиксируйте ограничение.

In [ ]:
results = []
for model_id in MODEL_IDS:
    config = AutoConfig.from_pretrained(model_id)
    with init_empty_weights(include_buffers=True):
        skeleton = AutoModelForCausalLM.from_config(config)
    n_params = sum(p.numel() for p in skeleton.parameters())
    results.append({'model': model_id, 'params': n_params,
                    'params_M': round(n_params/1e6, 1)})
    del skeleton
model_table = pd.DataFrame(results)
display(model_table)
# TODO: расположите модели по числу параметров и вычислите отношение самой большой к маленькой.

### Шаг 3. Приближённая оценка VRAM

Простейший нижний ориентир для **весов**: $P\times b$, где $P$ — количество параметров, $b$ — байт на параметр. При FP32 $b=4$, при FP16/BF16 $b=2$. Это **не пиковая память inference**: ещё нужны KV-cache, активации, буферы и место для CUDA.

При **полном fine-tuning с AdamW и типичной mixed precision конфигурацией** часто используют ориентир **~18 байт на параметр плюс память активаций** (состав зависит от способа хранения master weights/градиентов/состояний оптимизатора). Это не гарантированное требование для любого trainer. Для LoRA/QLoRA оценки принципиально другие: часть весов заморожена, но память активаций и другие затраты сохраняются.

Посчитайте **теоретический бюджет**, а затем ответьте: почему модель с весами 3 GiB может не поместиться при полном fine-tuning на GPU с 8 GiB VRAM?

In [ ]:
resource_table = model_table.copy()
resource_table['FP32_weights_GiB'] = resource_table.params * 4 / 2**30
resource_table['FP16_weights_GiB'] = resource_table.params * 2 / 2**30
resource_table['full_ft_18B_GiB'] = resource_table.params * 18 / 2**30
cols = ['FP32_weights_GiB', 'FP16_weights_GiB', 'full_ft_18B_GiB']
display(resource_table[['model', 'params_M'] + cols].round(2))
ax = resource_table.set_index('model')[cols].plot.bar(figsize=(9, 4))
ax.set_ylabel('GiB (грубая оценка)')
ax.set_title('Веса vs оценка полного обучения, без активаций')
plt.xticks(rotation=15, ha='right'); plt.tight_layout(); plt.show()
# TODO: добавьте линию уровня 16 GiB. Какие модели заведомо требуют оптимизации?

### Шаг 4. Проверяем оценку на загруженной модели

В отличие от `meta`-моделей, **ruGPT3-small уже загружена**, поэтому можем посмотреть реальные числа для её весов и GPU. Но помните: `max_memory_allocated()` измеряет выделенную PyTorch память во время замера, не полное потребление GPU всей системой; измерение зависит от длины входа, batch size и кеширования.

In [ ]:
actual_params = sum(p.numel() for p in gen_model.parameters())
actual_bytes = sum(p.numel() * p.element_size() for p in gen_model.parameters())
print('Параметров:', f'{actual_params:,}')
print('Память весов (GiB):', round(actual_bytes/2**30, 3))
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        _ = gen_model.generate(**encoded, max_new_tokens=12,
                               pad_token_id=gen_tok.eos_token_id)
    print('Пиковая allocated GPU память (GiB):',
          round(torch.cuda.max_memory_allocated()/2**30, 3))
else:
    print('Замер GPU пропущен: выполняется на CPU')
# TODO: сравните размер весов и пиковое потребление; объясните разницу.

**Вывод по ресурсам — заполните сами:**

| Модель | Параметры, млн | FP16 веса, GiB | Full FT, GiB без активаций | Реалистично на T4 16 GB? Почему? |
|---|---:|---:|---:|---|
| SmolLM2-135M | ... | ... | ... | ... |
| SmolLM2-360M | ... | ... | ... | ... |
| Qwen2.5-1.5B | ... | ... | ... | ... |

Для **реального планирования** нужно знать sequence length, batch size, optimizer, precision, gradient checkpointing, LoRA/full fine-tuning и резерв памяти. Не делайте вывод «влезет/не влезет» исключительно по колонке FP16 weights.

In [ ]:
# Освободите GPU после экспериментов с генерацией перед обучением NER.
import gc
del gen_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Практика 5. RoBERTa для NER: контраст с Decoder

RoBERTa формирует двунаправленные представления токенов, а `AutoModelForTokenClassification` присваивает токенам метки **BIO**. Это **не генерация текста**. В этом блоке используем небольшую англоязычную выборку [CoNLL-2003](https://huggingface.co/datasets/lhoestq/conll2003) с метками `PER`, `ORG`, `LOC`, `MISC`. Модель [DistilRoBERTa](https://huggingface.co/distilbert/distilroberta-base) компактнее оригинальной RoBERTa.

**BIO:** `B-PER` начинает сущность, `I-PER` продолжает, `O` означает токен вне сущности. Сложность: исходные метки относятся к словам, а токенизатор иногда разбивает слово на несколько подтокенов. Используем `word_ids()` и `-100` для токенов, не участвующих в loss. Материал: [Hugging Face — Token classification](https://huggingface.co/docs/transformers/main/tasks/token_classification).

In [ ]:
ner_data = load_dataset('lhoestq/conll2003')
tag_feature = ner_data['train'].features['ner_tags'].feature
labels = tag_feature.names if hasattr(tag_feature, 'names') else [
    'O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG',
    'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']
print('Метки:', labels)
print('Размеры частей:', {name: len(part) for name, part in ner_data.items()})
print('Пример:', ner_data['train'][0]['tokens'][:20])
print('BIO:', [labels[i] for i in ner_data['train'][0]['ner_tags'][:20]])
# TODO: посчитайте число примеров, содержащих хотя бы одну ORG.

### Шаг 1. Подготовить токенизатор и выровнять метки

**Заготовка:** сначала разобьём список слов на subtokens, затем каждому subtoken определим исходное слово через `word_ids(batch_index=i)`. Простая стратегия — метку ставим только на **первый** subtoken слова, остальные получают `-100`. Важно не присвоить метку специальным токенам.

In [ ]:
NER_MODEL = 'distilbert/distilroberta-base'
ner_tok = AutoTokenizer.from_pretrained(NER_MODEL, use_fast=True, add_prefix_space=True)

def tokenize_and_align_labels(batch):
    enc = ner_tok(batch['tokens'], is_split_into_words=True,
                  truncation=True, max_length=128)
    new_labels = []
    for i, word_labels in enumerate(batch['ner_tags']):
        word_ids = enc.word_ids(batch_index=i)
        previous_id = None
        row = []
        for word_id in word_ids:
            if word_id is None:
                row.append(-100)
            elif word_id != previous_id:
                row.append(word_labels[word_id])
            else:
                row.append(-100)  # TODO: объясните, зачем пропускаем прочие subtokens
            previous_id = word_id
        new_labels.append(row)
    enc['labels'] = new_labels
    return enc

train_small = ner_data['train'].shuffle(seed=SEED).select(range(500))
valid_small = ner_data['validation'].shuffle(seed=SEED).select(range(150))
train_enc = train_small.map(tokenize_and_align_labels, batched=True,
                            remove_columns=train_small.column_names)
valid_enc = valid_small.map(tokenize_and_align_labels, batched=True,
                            remove_columns=valid_small.column_names)
print('Число train / valid:', len(train_enc), len(valid_enc))
# TODO: выведите десять токенов и соответствующие метки для одного примера.

### Шаг 2. Дообучить модель на маленькой подвыборке

Используем 1 эпоху и небольшие batch size. Это **демонстрация pipeline**, а не стремление достичь высокого F1. Если нет GPU, обучение может быть медленным — можно сначала ограничиться 100 примерами.

**Заполните недостающие параметры**, опираясь на комментарии. `Trainer` вычисляет loss с учётом `-100`.

In [ ]:
id2label = dict(enumerate(labels))
label2id = {name: i for i, name in id2label.items()}
ner_model = AutoModelForTokenClassification.from_pretrained(
    NER_MODEL, num_labels=len(labels), id2label=id2label, label2id=label2id)
collator = DataCollatorForTokenClassification(tokenizer=ner_tok)
args = TrainingArguments(
    output_dir='ner_roberta_demo',
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=20,
    save_strategy='no',
    report_to='none',
)
trainer = Trainer(model=ner_model, args=args,
                  train_dataset=train_enc, eval_dataset=valid_enc,
                  data_collator=collator, processing_class=ner_tok)
# TODO: запустите trainer.train() и запишите loss по логам.

### Шаг 3. Посчитать entity-level F1 и посмотреть ошибки

Для NER обычная accuracy по токенам часто вводит в заблуждение: метка `O` встречается часто. `seqeval` оценивает правильность **типов и границ сущностей**. Нужно исключить позиции `-100`, восстановить имена BIO-классов и сравнить последовательности.

**Прогноз:** если модель угадала `B-PER`, но пропустила `I-PER`, будет ли целая сущность считаться корректной?

In [ ]:
import evaluate
metric = evaluate.load('seqeval')
# TODO: после trainer.train() выполните эти действия:
# pred = trainer.predict(valid_enc)
# pred_ids = pred.predictions.argmax(axis=-1)
# true_seq, pred_seq = [], []
# for row_pred, row_gold in zip(pred_ids, pred.label_ids):
#     indices = [i for i, gold in enumerate(row_gold) if gold != -100]
#     true_seq.append([labels[row_gold[i]] for i in indices])
#     pred_seq.append([labels[row_pred[i]] for i in indices])
# print(metric.compute(predictions=pred_seq, references=true_seq))

### Шаг 4. Покажите результат на собственной новости

Возьмите небольшое английское предложение с персоной, организацией и городом. Постройте `pipeline('token-classification', aggregation_strategy='simple')` на своей обученной модели и проверьте, какие сущности она выделяет. **Не полагайтесь только на overall F1:** найдите 1 удачный и 1 ошибочный пример. Модель обучалась всего на 500 примерах, поэтому ошибки ожидаемы.

In [ ]:
sample_text = 'Microsoft opened an office in London, and Satya Nadella spoke at the event.'
# TODO: после обучения создайте pipeline для NER с aggregation_strategy='simple'.
# ner_pipe = pipeline('token-classification', model=ner_model,
#                     tokenizer=ner_tok, aggregation_strategy='simple', device=0 if torch.cuda.is_available() else -1)
# entities = ner_pipe(sample_text)
# print(entities)

## Что нужно уметь показать после выполнения

1. **Causal mask:** сколько разрешённых элементов у маски длиной 6 и почему.
2. **Генерация:** топ-5 кандидатов следующего токена, их вероятности, два разных текста с одинаковым prompt и объяснение причины различий.
3. **Logits processing:** чему равна вероятность запрещённого токена и как изменились вероятности остальных; как меняется энтропия с температурой.
4. **Ресурсы:** сколько параметров у **трёх** моделей, сколько GiB требуют одни веса в FP16 и как получена приблизительная оценка полного fine-tuning.
5. **NER:** число примеров, training loss, entity-level F1 и один пример найденной/пропущенной сущности.

**Важно:** сравнивайте только результаты, полученные в одной и той же конфигурации; не путайте оценку памяти с реальным профилированием GPU.

## Дополнительные источники и открытые ноутбуки

- [The Illustrated GPT-2 — Jay Alammar](https://jalammar.github.io/illustrated-gpt2/) — анимированное и визуальное объяснение decoder-only архитектуры.
- [The Annotated Transformer — Harvard NLP](https://nlp.seas.harvard.edu/annotated-transformer/) — архитектурные компоненты и реализация.
- [Google Colab: autoregressive decoding](https://colab.research.google.com/github/prashantkul/learn-generative-ai/blob/main/07-autoregressive/notebook.ipynb) — как из next-token prediction получается цикл генерации.
- [Google Colab: decoding and sampling](https://colab.research.google.com/github/book-companion/blog-companion/blob/main/running-models/23-decoding-and-sampling.ipynb) — greedy, sampling, температура.
- [ARENA: Transformer from Scratch](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter1_transformer_interp/exercises/part1_transformer_from_scratch/1.1_Transformer_from_Scratch_exercises.ipynb) — дополнительная углублённая практика.
- [Hugging Face — Generation strategies](https://huggingface.co/docs/transformers/v4.49.0/generation_strategies) и [LogitsProcessor](https://huggingface.co/docs/transformers/main/internal/generation_utils).
- [Hugging Face — Model training anatomy](https://huggingface.co/docs/transformers/v4.37.0/en/model_memory_anatomy) и [Accelerate: init_empty_weights](https://huggingface.co/docs/accelerate/main/en/package_reference/big_modeling).
- [Hugging Face — Token classification](https://huggingface.co/docs/transformers/main/tasks/token_classification).
- [Hugging Face — How caching works](https://huggingface.co/docs/transformers/main/cache_explanation) — формы K/V, prefill, decode и кеширование.
- [Hugging Face — Cache strategies](https://huggingface.co/docs/transformers/main/kv_cache) — DynamicCache, StaticCache, ограничения памяти.
